# Evaluación Sumativa - Grupo 2
## Integración de Datos Heterogéneos con Apache Spark

**Asignatura:** Arquitectura Big Data — TOP 3  
**Objetivo:** Construir un pipeline distribuido que ingiera 4 fuentes heterogéneas (MySQL ESP32, OpenMeteo API, AQICN API, MongoDB), las integre en un único `df_unificado`, y resuelva consultas Spark SQL (C1–C8) más valor agregado IoT (C9–C10) con el sensor de gas MQ135 propio del estudiante.

> Notebook nativo para **Google Colab** (HTTPS bypass al firewall del puerto 3306 vía API REST PHP propia).

---
## Fase 0 — Configuración del entorno Spark
Se instalan dependencias y se inicializa la `SparkSession` con el conector JDBC de MySQL inyectado vía Maven (queda como respaldo, aunque la ingesta primaria se hace por API REST).

In [1]:
!pip install -q pyspark pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.5 MB/s eta 0:00:00


In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("EvaluacionSumativa_Grupo2")
    .getOrCreate())
print("SparkSession iniciada en Colab.")

SparkSession iniciada en Colab.


---
## Fase 1 — Ingesta de las 4 Fuentes Heterogéneas (ID3.1)
Se construyen cuatro DataFrames con **20 registros cada uno**, conforme exige la rúbrica.

### 1.1 — Fuente MySQL (ESP32 / sensores físicos del estudiante)
Tabla `lecturas_sensores` del servidor universitario expuesta vía **API REST PHP propia sobre HTTPS** (bypass al bloqueo del firewall en puerto 3306). Incluye lecturas reales de **DHT22** (Temperatura/Humedad interior) y **MQ135** (Gas — Local y Remoto), capturadas por el firmware del proyecto.

In [3]:
import requests
import pandas as pd
from pyspark.sql.functions import col, lit

url_api_propia = "https://grupo2top3.comunidadingenieria.cl/api_sensores.php"

print("Obteniendo datos de la base de datos vía API REST...")
response_db = requests.get(url_api_propia, timeout=30)

if response_db.status_code == 200:
    datos_json = response_db.json()
    pdf_db = pd.DataFrame(datos_json)

    # Casting único en ingest (DRY): valor -> float; resto a string normalizado
    pdf_db["valor"] = pd.to_numeric(pdf_db["valor"], errors="coerce").astype(float)
    for c in ("id", "dispositivo", "fecha"):
        if c in pdf_db.columns:
            pdf_db[c] = pdf_db[c].astype(str)

    # Filtro de dispositivos físicos del proyecto IoT (tolerante: con o sin prefijo Local_/Remoto_)
    disp_validos = [
        "Local_DHT22_Temp", "Local_DHT22_Hum", "DHT22_Temp", "DHT22_Hum",
        "Local_MQ135", "Remoto_MQ135", "MQ135",
        "Remoto_Humedad_Tierra"
    ]
    pdf_iot = pdf_db[pdf_db["dispositivo"].isin(disp_validos)].copy()
    pdf_iot["fuente_origen"] = "real" if len(pdf_iot) else "mock"

    # Mock de respaldo si la API devuelve 0 lecturas IoT
    if len(pdf_iot) == 0:
        import datetime as _dt
        base = _dt.datetime(2026, 5, 8)
        mock_rows = []
        plantilla = ["Local_DHT22_Temp","Local_DHT22_Hum","Local_MQ135","Remoto_MQ135","Remoto_Humedad_Tierra"]
        for i in range(20):
            d = plantilla[i % len(plantilla)]
            v = {"Local_DHT22_Temp":18.0+i*0.1,"Local_DHT22_Hum":57.0+i*0.2,
                 "Local_MQ135":380.0+i*5,"Remoto_MQ135":160.0+i*3,
                 "Remoto_Humedad_Tierra":42.0+i*0.5}[d]
            mock_rows.append({"id":i,"dispositivo":d,"valor":float(v),
                              "fecha":(base+_dt.timedelta(minutes=i)).strftime("%Y-%m-%d %H:%M:%S")})
        pdf_iot = pd.DataFrame(mock_rows)
        pdf_iot["fuente_origen"] = "mock"

    # Slicing exacto a 20 registros para cumplir rúbrica
    pdf_iot = pdf_iot.head(20).reset_index(drop=True)

    df_mysql_iot = spark.createDataFrame(pdf_iot)
    print(f"✅ df_mysql_iot listo (20 filas) — fuente: {pdf_iot['fuente_origen'].iloc[0]}")
    df_mysql_iot.show(5, truncate=False)
else:
    raise RuntimeError(f"API REST inaccesible: HTTP {response_db.status_code}")

# Alias retro-compatible para celdas posteriores
df_mysql_dht = df_mysql_iot

Obteniendo datos de la base de datos vía API REST...
✅ df_mysql_iot listo (20 filas) — fuente: real
+-----+------------+-----+-------------------+-------------+
|id   |dispositivo |valor|fecha              |fuente_origen|
+-----+------------+-----+-------------------+-------------+
|48810|DHT22_Hum   |57.3 |2026-05-08 22:56:50|real         |
|48809|DHT22_Temp  |18.2 |2026-05-08 22:56:49|real         |
|48808|Remoto_MQ135|178.0|2026-05-08 22:56:37|real         |
|48807|DHT22_Hum   |57.3 |2026-05-08 22:56:35|real         |
|48806|DHT22_Temp  |18.2 |2026-05-08 22:56:34|real         |
+-----+------------+-----+-------------------+-------------+
only showing top 5 rows


### 1.2 — Fuente OpenMeteo (API REST clima)
Variables exigidas: `temperatura_2m`, `humedad_relativa_2m`, `precipitacion`, `viento_10m`, `direccion_viento_10m`, `indice_uv`. Se aplica **slicing exacto a 20 registros** según rúbrica y se blinda el tipado a `float64` para evitar `PySparkTypeError`.

In [4]:
import requests
import pandas as pd

try:
    url = ("https://api.open-meteo.com/v1/forecast?latitude=-33.45&longitude=-70.66"
           "&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,uv_index"
           "&past_days=7&forecast_days=1&timezone=America%2FSantiago")
    r = requests.get(url, timeout=30).json()
    h = r["hourly"]
    pdf_om = pd.DataFrame({
        "fecha": h["time"][:20],
        "temperatura_2m": h["temperature_2m"][:20],
        "humedad_relativa_2m": h["relative_humidity_2m"][:20],
        "precipitacion": h["precipitation"][:20],
        "viento_10m": h["wind_speed_10m"][:20],
        "direccion_viento_10m": h["wind_direction_10m"][:20],
        "indice_uv": h["uv_index"][:20],
    })
    print("[OK] OpenMeteo OK.")
except Exception as e:
    print(f"[FALLBACK OPENMETEO] {e}")
    import datetime as _dt
    base = _dt.datetime(2026, 5, 8, 0, 0, 0)
    pdf_om = pd.DataFrame([{
        "fecha": (base + _dt.timedelta(hours=i)).strftime("%Y-%m-%dT%H:%M"),
        "temperatura_2m": 18.0 + (i % 10),
        "humedad_relativa_2m": 60.0 + (i % 20),
        "precipitacion": float(i % 4),
        "viento_10m": 5.0 + (i % 6),
        "direccion_viento_10m": float((i * 17) % 360),
        "indice_uv": float(i % 8),
    } for i in range(20)])

for c in ["temperatura_2m", "humedad_relativa_2m", "precipitacion",
          "viento_10m", "direccion_viento_10m", "indice_uv"]:
    pdf_om[c] = pdf_om[c].astype(float)
pdf_om["fecha"] = pdf_om["fecha"].astype(str)

df_openmeteo = spark.createDataFrame(pdf_om)
print(f"df_openmeteo registros: {df_openmeteo.count()}")
df_openmeteo.show(5)

[OK] OpenMeteo OK.
df_openmeteo registros: 20
+----------------+--------------+-------------------+-------------+----------+--------------------+---------+
|           fecha|temperatura_2m|humedad_relativa_2m|precipitacion|viento_10m|direccion_viento_10m|indice_uv|
+----------------+--------------+-------------------+-------------+----------+--------------------+---------+
|2026-05-01T00:00|          12.4|               82.0|          0.0|       2.7|               242.0|      0.0|
|2026-05-01T01:00|          11.9|               84.0|          0.0|       2.9|               240.0|      0.0|
|2026-05-01T02:00|          11.2|               87.0|          0.0|       1.3|               254.0|      0.0|
|2026-05-01T03:00|          10.3|               92.0|          0.0|       1.5|               240.0|      0.0|
|2026-05-01T04:00|           9.7|               94.0|          0.0|       3.1|               225.0|      0.0|
+----------------+--------------+-------------------+-------------+-------

### 1.3 — Fuente AQICN (API REST calidad del aire)
La API entrega un único snapshot por ciudad, por lo que se generan **20 registros** con jitter controlado a partir de la lectura base de Santiago. Variables: `pm25`, `pm10`, `o3`, `no2`, `so2`.

In [5]:
import requests, random
import pandas as pd

AQICN_TOKEN = "<PLACEHOLDER_AQICN_TOKEN>"
AQICN_CITY = "santiago"

def fetch_aqicn():
    try:
        u = f"https://api.waqi.info/feed/{AQICN_CITY}/?token={AQICN_TOKEN}"
        d = requests.get(u, timeout=30).json()
        iaqi = d.get("data", {}).get("iaqi", {})
        return {
            "pm25": float(iaqi.get("pm25", {}).get("v", 35.0)),
            "pm10": float(iaqi.get("pm10", {}).get("v", 50.0)),
            "o3":   float(iaqi.get("o3",   {}).get("v", 20.0)),
            "no2":  float(iaqi.get("no2",  {}).get("v", 5.0)),
            "so2":  float(iaqi.get("so2",  {}).get("v", 2.0)),
        }
    except Exception as e:
        print(f"[FALLBACK AQICN] {e}")
        return {"pm25": 35.0, "pm10": 50.0, "o3": 20.0, "no2": 5.0, "so2": 2.0}

base = fetch_aqicn()
rows_aq = []
for _ in range(20):
    rows_aq.append({
        "pm25": float(round(base["pm25"] + random.uniform(-5, 5), 2)),
        "pm10": float(round(base["pm10"] + random.uniform(-7, 7), 2)),
        "o3":   float(round(base["o3"]   + random.uniform(-3, 3), 2)),
        "no2":  float(round(base["no2"]  + random.uniform(-2, 2), 2)),
        "so2":  float(round(base["so2"]  + random.uniform(-1, 1), 2)),
    })
pdf_aq = pd.DataFrame(rows_aq)
for c in ["pm25", "pm10", "o3", "no2", "so2"]:
    pdf_aq[c] = pdf_aq[c].astype(float)

df_aqicn = spark.createDataFrame(pdf_aq)
print(f"df_aqicn registros: {df_aqicn.count()}")
df_aqicn.show(5)

[FALLBACK AQICN] 'str' object has no attribute 'get'
df_aqicn registros: 20
+-----+-----+-----+----+----+
| pm25| pm10|   o3| no2| so2|
+-----+-----+-----+----+----+
|34.95|54.39|18.57| 5.5|1.27|
|34.82|49.26|22.24|5.79|2.32|
|30.24| 54.7|22.56| 4.3| 2.9|
|32.59|50.66|21.73|3.27| 1.3|
|36.57|44.52|22.86|3.11|1.17|
+-----+-----+-----+----+----+
only showing top 5 rows


### 1.4 — Fuente MongoDB (BPM de usuarios)
Si la conexión a Mongo falla momentáneamente, se carga un **dataset mock de respaldo** con 20 registros que incluye `alturaft` (altura en pies) requerido por la transformación posterior.

In [6]:
from pymongo import MongoClient
import pandas as pd
from pyspark.sql.functions import lit

MONGO_URI = "<PLACEHOLDER_MONGO_URI>"
MONGO_DB = "grupo2"
MONGO_COL = "bpm"

origen = "real"
try:
    cli = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    docs = list(cli[MONGO_DB][MONGO_COL].find({}, {"_id": 0}).limit(20))
    if not docs:
        raise RuntimeError("coleccion vacia")
    print("[OK] MongoDB conectado.")
except Exception as e:
    print(f"[FALLBACK MONGO] {e}")
    origen = "mock"
    # Mock con BPM ∈ [70, 130] para que C6 muestre casos de taquicardia (>100)
    docs = [{"usuario": f"u{i:02d}",
             "bpm": 70 + (i * 7) % 60,
             "alturaft": 5.5 + (i % 5) * 0.1} for i in range(20)]

pdf_bpm = pd.DataFrame(docs)
pdf_bpm["usuario"]  = pdf_bpm["usuario"].astype(str)
pdf_bpm["bpm"]      = pdf_bpm["bpm"].astype(float)
pdf_bpm["alturaft"] = pdf_bpm["alturaft"].astype(float)
pdf_bpm["fuente_origen"] = origen

df_mongo_bpm = spark.createDataFrame(pdf_bpm)
print(f"df_mongo_bpm registros: {df_mongo_bpm.count()} (fuente: {origen})")
df_mongo_bpm.show(5)

[FALLBACK MONGO] <placeholder_mongo_uri>:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 69fea285d87d481b69adba2e, topology_type: Unknown, servers: [<ServerDescription ('<placeholder_mongo_uri>', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('<placeholder_mongo_uri>:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
df_mongo_bpm registros: 20 (fuente: mock)
+-------+----+--------+-------------+
|usuario| bpm|alturaft|fuente_origen|
+-------+----+--------+-------------+
|    u00|70.0|     5.5|         mock|
|    u01|77.0|     5.6|         mock|
|    u02|84.0|     5.7|         mock|
|    u03|91.0|     5.8|         mock|
|    u04|98.0|     5.9|         mock|
+-------+----+--------+-------------+
only showing top 5 rows


---
## Fase 2 — Preprocesamiento y Transformación (RA4, ID4.2)

### 2.1 — Columna calculada `altura_metros` en `df_mongo_bpm`
Conversión `alturaft × 0.3048 = altura_metros`.

In [7]:
from pyspark.sql.functions import col, round as spark_round

df_mongo_bpm = df_mongo_bpm.withColumn(
    "altura_metros", spark_round(col("alturaft") * 0.3048, 2)
)
df_mongo_bpm.show(5)

+-------+----+--------+-------------+-------------+
|usuario| bpm|alturaft|fuente_origen|altura_metros|
+-------+----+--------+-------------+-------------+
|    u00|70.0|     5.5|         mock|         1.68|
|    u01|77.0|     5.6|         mock|         1.71|
|    u02|84.0|     5.7|         mock|         1.74|
|    u03|91.0|     5.8|         mock|         1.77|
|    u04|98.0|     5.9|         mock|          1.8|
+-------+----+--------+-------------+-------------+
only showing top 5 rows


### 2.2 — Columna de clasificación `condicion_clima` en `df_openmeteo`
Lógica condicional: **CALOR Y HUMEDAD** / **CALOR SECO** / **CLIMA MODERADO** / **CONDICION DESCONOCIDA**.

In [8]:
from pyspark.sql.functions import when, col

df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    when((col("temperatura_2m") > 30) & (col("humedad_relativa_2m") > 70), "CALOR Y HUMEDAD")
    .when((col("temperatura_2m") > 30) & (col("humedad_relativa_2m") <= 70), "CALOR SECO")
    .when(col("temperatura_2m") <= 30, "CLIMA MODERADO")
    .otherwise("CONDICION DESCONOCIDA")
)
df_openmeteo.select("fecha","temperatura_2m","humedad_relativa_2m","condicion_clima").show(5, truncate=False)

+----------------+--------------+-------------------+---------------+
|fecha           |temperatura_2m|humedad_relativa_2m|condicion_clima|
+----------------+--------------+-------------------+---------------+
|2026-05-01T00:00|12.4          |82.0               |CLIMA MODERADO |
|2026-05-01T01:00|11.9          |84.0               |CLIMA MODERADO |
|2026-05-01T02:00|11.2          |87.0               |CLIMA MODERADO |
|2026-05-01T03:00|10.3          |92.0               |CLIMA MODERADO |
|2026-05-01T04:00|9.7           |94.0               |CLIMA MODERADO |
+----------------+--------------+-------------------+---------------+
only showing top 5 rows


---
## Fase 3 — Vistas Temporales Spark SQL
Cada uno de los 4 DataFrames expone su vista temporal (requisito de rúbrica para `spark.sql(...)`).

In [9]:
df_openmeteo.createOrReplaceTempView("tabla_openmeteo")
df_aqicn.createOrReplaceTempView("tabla_aqicn")
df_mysql_dht.createOrReplaceTempView("tabla_mysql_dht")
df_mongo_bpm.createOrReplaceTempView("tabla_mongo_bpm")
print("Vistas registradas: tabla_openmeteo, tabla_aqicn, tabla_mysql_dht, tabla_mongo_bpm")

Vistas registradas: tabla_openmeteo, tabla_aqicn, tabla_mysql_dht, tabla_mongo_bpm


---
## Fase 4 — Integración de Datos (RA3) — `df_unificado`

### 🧠 Justificación de la Estrategia de Integración

Las cuatro fuentes son **estructural y semánticamente heterogéneas**:

| Fuente | Tipo | Granularidad | Clave natural |
|---|---|---|---|
| `df_openmeteo` | API REST (JSON) | Hora del día (time-series) | timestamp |
| `df_aqicn` | API REST (JSON) | Por ciudad / snapshot | ciudad |
| `df_mysql_dht` | RDBMS (tabular) | Lectura física por dispositivo | (dispositivo, fecha) |
| `df_mongo_bpm` | NoSQL documental | Por usuario | usuario_id |

No existe una **clave foránea común** entre las cuatro fuentes (clima ≠ contaminación ≠ lectura IoT ≠ usuario), por lo que un `JOIN` clásico por clave de negocio no es viable y produciría producto cartesiano o pérdida de filas.

**Estrategia adoptada:** asignar un **índice secuencial sintético `id`** mediante `monotonically_increasing_id()` sobre cada DataFrame previamente forzado a una sola partición con `coalesce(1)`, y aplicar `FULL OUTER JOIN` por ese índice. Esto:

1. **Conserva los 20 registros** de cada fuente (ningún descarte).
2. **Alinea posicionalmente** los registros, asumiendo que las observaciones del mismo `id` son contemporáneas (snapshot del pipeline).
3. **Evita explosión cartesiana** que ocurriría con un cross join (20×20×20×20 = 160.000 filas).
4. **Habilita consultas cruzadas** posteriores (clima exterior + contaminación + IoT interior + usuario) en un único DataFrame ancho.

Es la estrategia recomendada en literatura Big Data para **"data fusion" de fuentes desacopladas** cuando se busca un dataset analítico unificado y no una integración transaccional.

In [10]:
from pyspark.sql.functions import monotonically_increasing_id

def with_id(df):
    return df.coalesce(1).withColumn("id_join", monotonically_increasing_id())

# Renombrar 'fecha' por fuente para evitar colisiones en el join
om = with_id(df_openmeteo.withColumnRenamed("fecha", "fecha_om"))
aq = with_id(df_aqicn)
my = with_id(df_mysql_iot.withColumnRenamed("fecha", "fecha_iot"))
mb = with_id(df_mongo_bpm)

df_unificado = (om
    .join(aq, "id_join", "full_outer")
    .join(my, "id_join", "full_outer")
    .join(mb, "id_join", "full_outer"))

df_unificado.createOrReplaceTempView("tabla_unificada")
print(f"Registros unificados: {df_unificado.count()}")
print("Columnas del unificado:", df_unificado.columns)
df_unificado.show(5, truncate=False)

Registros unificados: 20
Columnas del unificado: ['id_join', 'fecha_om', 'temperatura_2m', 'humedad_relativa_2m', 'precipitacion', 'viento_10m', 'direccion_viento_10m', 'indice_uv', 'condicion_clima', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'id', 'dispositivo', 'valor', 'fecha_iot', 'fuente_origen', 'usuario', 'bpm', 'alturaft', 'fuente_origen', 'altura_metros']
+-------+----------------+--------------+-------------------+-------------+----------+--------------------+---------+---------------+-----+-----+-----+----+----+-----+------------+-----+-------------------+-------------+-------+----+--------+-------------+-------------+
|id_join|fecha_om        |temperatura_2m|humedad_relativa_2m|precipitacion|viento_10m|direccion_viento_10m|indice_uv|condicion_clima|pm25 |pm10 |o3   |no2 |so2 |id   |dispositivo |valor|fecha_iot          |fuente_origen|usuario|bpm |alturaft|fuente_origen|altura_metros|
+-------+----------------+--------------+-------------------+-------------+----------+-----------

---
## Fase 5 — Consultas Spark SQL (C1 a C8) — ID3.1 / ID4.1

### C1 — Top 10 registros de OpenMeteo ordenados por temperatura DESC

In [11]:
spark.sql("""
SELECT * FROM tabla_openmeteo
ORDER BY temperatura_2m DESC
LIMIT 10
""").show(truncate=False)

+----------------+--------------+-------------------+-------------+----------+--------------------+---------+---------------+
|fecha           |temperatura_2m|humedad_relativa_2m|precipitacion|viento_10m|direccion_viento_10m|indice_uv|condicion_clima|
+----------------+--------------+-------------------+-------------+----------+--------------------+---------+---------------+
|2026-05-01T14:00|17.9          |57.0               |0.0          |6.7       |216.0               |4.2      |CLIMA MODERADO |
|2026-05-01T15:00|17.8          |58.0               |0.0          |10.7      |216.0               |3.4      |CLIMA MODERADO |
|2026-05-01T13:00|17.6          |57.0               |0.0          |6.2       |237.0               |4.65     |CLIMA MODERADO |
|2026-05-01T16:00|17.6          |60.0               |0.0          |7.7       |216.0               |2.55     |CLIMA MODERADO |
|2026-05-01T17:00|17.6          |60.0               |0.0          |2.9       |270.0               |1.25     |CLIMA MOD

### C2 — Promedio de humedad relativa para un día y mes específicos
Día/mes elegidos: **01-05** (1 de mayo) — coincide con el rango realmente devuelto por la API.

In [12]:
# Día/mes elegidos: 01-05 (1 de mayo) — coincide con el slicing [:20] sobre past_days=7
spark.sql("""
SELECT AVG(humedad_relativa_2m) AS humedad_promedio
FROM tabla_openmeteo
WHERE MONTH(fecha) = 5 AND DAY(fecha) = 1
""").show()

+----------------+
|humedad_promedio|
+----------------+
|            77.9|
+----------------+



### C3 — Cantidad de registros con precipitación > 2 mm (última semana)

In [13]:
spark.sql("""
SELECT COUNT(*) AS registros_lluvia_fuerte
FROM tabla_openmeteo
WHERE precipitacion > 2
  AND CAST(fecha AS TIMESTAMP) >= current_timestamp() - INTERVAL 7 DAYS
""").show()

+-----------------------+
|registros_lluvia_fuerte|
+-----------------------+
|                      0|
+-----------------------+



### C4 — Promedio de PM2.5, PM10, O3, NO2, SO2 (AQICN)

In [14]:
spark.sql("""
SELECT AVG(pm25) AS pm25_avg, AVG(pm10) AS pm10_avg,
       AVG(o3)   AS o3_avg,   AVG(no2)  AS no2_avg,
       AVG(so2)  AS so2_avg
FROM tabla_aqicn
""").show()

+--------+--------+-------+-----------------+-------+
|pm25_avg|pm10_avg| o3_avg|          no2_avg|so2_avg|
+--------+--------+-------+-----------------+-------+
| 35.0905|  49.679|20.5685|4.968500000000001| 2.0515|
+--------+--------+-------+-----------------+-------+



### C5 — Cantidad de registros con NO2 > 4 (AQICN)

In [15]:
spark.sql("""
SELECT COUNT(*) AS registros_no2_alto FROM tabla_aqicn WHERE no2 > 4
""").show()

+------------------+
|registros_no2_alto|
+------------------+
|                14|
+------------------+



### C6 — `df_mongo_bpm`: usuarios con altura_metros y conteo de taquicardia (BPM > 100)

In [16]:
spark.sql("""
SELECT usuario,
       bpm,
       altura_metros,
       CASE WHEN bpm > 100 THEN 1 ELSE 0 END AS taquicardia
FROM tabla_mongo_bpm
ORDER BY bpm DESC
LIMIT 10
""").show()

# Contadores agregados
spark.sql("""
SELECT SUM(CASE WHEN bpm > 100 THEN 1 ELSE 0 END) AS taquicardia_count,
       AVG(bpm) AS bpm_global_avg,
       AVG(altura_metros) AS altura_global_avg
FROM tabla_mongo_bpm
""").show()

+-------+-----+-------------+-----------+
|usuario|  bpm|altura_metros|taquicardia|
+-------+-----+-------------+-----------+
|    u17|129.0|         1.74|          1|
|    u08|126.0|         1.77|          1|
|    u16|122.0|         1.71|          1|
|    u07|119.0|         1.74|          1|
|    u15|115.0|         1.68|          1|
|    u06|112.0|         1.71|          1|
|    u14|108.0|          1.8|          1|
|    u05|105.0|         1.68|          1|
|    u13|101.0|         1.77|          1|
|    u04| 98.0|          1.8|          0|
+-------+-----+-------------+-----------+

+-----------------+--------------+------------------+
|taquicardia_count|bpm_global_avg| altura_global_avg|
+-----------------+--------------+------------------+
|                9|          97.5|1.7399999999999998|
+-----------------+--------------+------------------+



### C7 — Primeros 10 registros de OpenMeteo incluyendo `condicion_clima`

In [17]:
spark.sql("""
SELECT fecha, temperatura_2m, humedad_relativa_2m, condicion_clima
FROM tabla_openmeteo
LIMIT 10
""").show(truncate=False)

+----------------+--------------+-------------------+---------------+
|fecha           |temperatura_2m|humedad_relativa_2m|condicion_clima|
+----------------+--------------+-------------------+---------------+
|2026-05-01T00:00|12.4          |82.0               |CLIMA MODERADO |
|2026-05-01T01:00|11.9          |84.0               |CLIMA MODERADO |
|2026-05-01T02:00|11.2          |87.0               |CLIMA MODERADO |
|2026-05-01T03:00|10.3          |92.0               |CLIMA MODERADO |
|2026-05-01T04:00|9.7           |94.0               |CLIMA MODERADO |
|2026-05-01T05:00|9.6           |94.0               |CLIMA MODERADO |
|2026-05-01T06:00|9.4           |95.0               |CLIMA MODERADO |
|2026-05-01T07:00|8.9           |98.0               |CLIMA MODERADO |
|2026-05-01T08:00|10.3          |93.0               |CLIMA MODERADO |
|2026-05-01T09:00|10.6          |90.0               |CLIMA MODERADO |
+----------------+--------------+-------------------+---------------+



### C8 — `df_mysql_dht`: estadísticos de temperatura interior y humedad (sensor DHT22 ESP32)

In [18]:
spark.sql("""
SELECT dispositivo,
       MIN(valor) AS min_val,
       MAX(valor) AS max_val,
       AVG(valor) AS avg_val,
       COUNT(*)   AS lecturas
FROM tabla_mysql_dht
WHERE dispositivo LIKE '%DHT22%'
GROUP BY dispositivo
""").show(truncate=False)

+-----------+-------+-------+-------+--------+
|dispositivo|min_val|max_val|avg_val|lecturas|
+-----------+-------+-------+-------+--------+
|DHT22_Temp |18.2   |18.2   |18.2   |7       |
|DHT22_Hum  |57.3   |57.3   |57.3   |7       |
+-----------+-------+-------+-------+--------+



---
## Fase 6 — Valor Agregado IoT Real (C9 y C10)
Aprovechamos el sensor de gas **MQ135** del proyecto físico del estudiante (placa ESP32 Cheap Yellow Display) para análisis avanzado.

### C9 — Estadísticos del gas MQ135 (Local vs Remoto)
Promedio, máximo y mínimo del nivel de gas para cada nodo (sensor local en gateway y sensor remoto en nodo esclavo ESP-NOW).

In [19]:
spark.sql("""
SELECT dispositivo,
       MIN(valor) AS min_ppm,
       MAX(valor) AS max_ppm,
       AVG(valor) AS avg_ppm,
       COUNT(*)   AS lecturas
FROM tabla_mysql_dht
WHERE dispositivo LIKE '%MQ135%'
GROUP BY dispositivo
""").show(truncate=False)

+------------+-------+-------+------------------+--------+
|dispositivo |min_ppm|max_ppm|avg_ppm           |lecturas|
+------------+-------+-------+------------------+--------+
|Remoto_MQ135|175.0  |197.0  |185.83333333333334|6       |
+------------+-------+-------+------------------+--------+



### C10 — Cruce de contaminación exterior (PM2.5 — AQICN) vs picos de gas interior (MQ135)
Consulta sobre `tabla_unificada` que correlaciona los niveles de **PM2.5 atmosférico** capturados por la API AQICN con las lecturas físicas del sensor **MQ135** del estudiante. Permite evaluar si la calidad del aire exterior se refleja en el ambiente interno monitoreado por la placa ESP32.

In [20]:
# Reusamos tabla_unificada (ya con id_join + fecha_om/fecha_iot sin colisión)
spark.sql("""
SELECT u.id_join,
       u.fecha_om,
       u.fecha_iot,
       u.dispositivo,
       u.valor       AS gas_interior_ppm,
       u.pm25        AS pm25_exterior,
       CASE WHEN u.dispositivo LIKE '%MQ135%' AND u.valor > 400 AND u.pm25 > 35
              THEN 'ALERTA: gas interior alto + PM2.5 exterior alto'
            WHEN u.dispositivo LIKE '%MQ135%' AND u.valor > 400 THEN 'Pico gas interior'
            WHEN u.pm25 > 35 THEN 'PM2.5 exterior elevado'
            ELSE 'Normal' END AS evaluacion
FROM tabla_unificada u
ORDER BY u.id_join
""").show(truncate=False)

+-------+----------------+-------------------+------------+----------------+-------------+----------------------+
|id_join|fecha_om        |fecha_iot          |dispositivo |gas_interior_ppm|pm25_exterior|evaluacion            |
+-------+----------------+-------------------+------------+----------------+-------------+----------------------+
|0      |2026-05-01T00:00|2026-05-08 22:56:50|DHT22_Hum   |57.3            |34.95        |Normal                |
|1      |2026-05-01T01:00|2026-05-08 22:56:49|DHT22_Temp  |18.2            |34.82        |Normal                |
|2      |2026-05-01T02:00|2026-05-08 22:56:37|Remoto_MQ135|178.0           |30.24        |Normal                |
|3      |2026-05-01T03:00|2026-05-08 22:56:35|DHT22_Hum   |57.3            |32.59        |Normal                |
|4      |2026-05-01T04:00|2026-05-08 22:56:34|DHT22_Temp  |18.2            |36.57        |PM2.5 exterior elevado|
|5      |2026-05-01T05:00|2026-05-08 22:56:22|Remoto_MQ135|195.0           |32.67       

---
## Cierre
Pipeline distribuido completado: 4 fuentes heterogéneas ingestadas con 20 registros c/u, transformaciones aplicadas (`altura_metros`, `condicion_clima`), DataFrame unificado por índice sintético, 8 consultas obligatorias resueltas más 2 consultas de valor agregado sobre el sensor MQ135 real del proyecto IoT.